# HR Policy Assistant



### Import all packages


In [4]:
import os 
from dotenv import load_dotenv

# langchain framwork packages
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveJsonSplitter
from langchain_community.embeddings import JinaEmbeddings

In [5]:
load_dotenv()

True

### Load ENV Data

In [6]:
groq_api_key=os.getenv("GROQ_API_KEY")
jina_api_key=os.getenv("JINA_API_KEY")


# PHASE-1: DATA INGESTION PIPELINE


### Loading Data

In [47]:
Data_Path=os.path.join("data","dummy_data.txt")

loader=TextLoader(Data_Path,encoding="utf-8")
documents=loader.load()
# print(documents[0].page_content)
# print(documents[0].metadata)
print(f"Loaded File : {Data_Path}")
print(f"Number of Document : {len(documents)}")
print(f"Total characters in document : {len(documents[0].page_content)}")
print("\n------------Previw of first 300 characters -------------")
print(documents[0].page_content[:300])

Loaded File : data\dummy_data.txt
Number of Document : 1
Total characters in document : 44522

------------Previw of first 300 characters -------------
TECHNOVA SOLUTIONS PVT. LTD.
EMPLOYEE HR POLICY HANDBOOK

Document: Employee HR Policy Handbook
Version: 3.0
Effective Date: January 1, 2026

IMPORTANT:
This is fictional content created for RAG system practice and demonstration. It is not a real company's HR policy.



#### LANGCHAIN DOCUMENT 

Langchain processes everything in form of documents 


DOCUMENTS : 

PAGE CONTENT -- the actual data 

METADATA  - extra information about the data like souce matlab kaha se data fetch kiya


In [48]:
len(documents)

1

In [51]:
print(documents[0].page_content)

TECHNOVA SOLUTIONS PVT. LTD.
EMPLOYEE HR POLICY HANDBOOK

Document: Employee HR Policy Handbook
Version: 3.0
Effective Date: January 1, 2026

IMPORTANT:
This is fictional content created for RAG system practice and demonstration. It is not a real company's HR policy.

1. COMPANY OVERVIEW

TechNova Solutions Pvt. Ltd. is a fictional technology company providing software development, cloud computing, artificial intelligence, cybersecurity, and consulting services.

The company employs full-time employees, part-time employees, interns, contractors, and temporary workers.

The purpose of this handbook is to define workplace rules, employee benefits, responsibilities, leave policies, attendance requirements, compensation practices, and employee conduct standards.

2. EMPLOYMENT TYPES

2.1 Full-Time Employee

A full-time employee normally works 40 hours per week.

Full-time employees are eligible for benefits according to their employment agreement and applicable company policies.

2.2 Part-

In [52]:
print(documents[0].metadata)

{'source': 'data\\dummy_data.txt'}


### SPLITTING DATA

In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks=text_splitter.split_documents(documents)
print(chunks)


[Document(metadata={'source': 'data\\dummy_data.txt'}, page_content="TECHNOVA SOLUTIONS PVT. LTD.\nEMPLOYEE HR POLICY HANDBOOK\n\nDocument: Employee HR Policy Handbook\nVersion: 3.0\nEffective Date: January 1, 2026\n\nIMPORTANT:\nThis is fictional content created for RAG system practice and demonstration. It is not a real company's HR policy.\n\n============================================================\n1. COMPANY OVERVIEW\n============================================================"), Document(metadata={'source': 'data\\dummy_data.txt'}, page_content='TechNova Solutions Pvt. Ltd. is a fictional technology company providing software development, cloud computing, artificial intelligence, cybersecurity, and consulting services.\n\nThe company employs full-time employees, part-time employees, interns, contractors, and temporary workers.\n\nThe purpose of this handbook is to define workplace rules, employee benefits, responsibilities, leave policies, attendance requirements, compensati

In [54]:
print(len(chunks))

106


In [55]:
print(chunks[2].page_content)

2. EMPLOYMENT TYPES

2.1 Full-Time Employee

A full-time employee normally works 40 hours per week.

Full-time employees are eligible for benefits according to their employment agreement and applicable company policies.

2.2 Part-Time Employee

A part-time employee works fewer than 40 hours per week.


### EMBEDD OUR DATA

In [56]:
from langchain_community.embeddings import JinaEmbeddings

embedding_model=JinaEmbeddings(
    model_name="jina-embeddings-v2-base-en"
)
print("EMB MODEL READY THE NAME IS ",embedding_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


### STRORE DATA IN VECTOR DB-FAISS

In [57]:
from langchain_community.vectorstores import FAISS

vector_store=FAISS.from_documents(chunks,embedding_model);

print("CHUNK ARE STORED", vector_store.index.ntotal)

CHUNK ARE STORED 106


### SEARCH QUERY

In [58]:
test_query="how many sick leaves employee get"

top_match=vector_store.similarity_search(test_query,k=2)
print(f"Query: {test_query}\n")

for i,match in enumerate(top_match,start=1):
    print(f"------Match {i}------")
    print(match.page_content)
    print()

Query: how many sick leaves employee get

------Match 1------
13. SICK LEAVE

Sick leave may be used when an employee is unable to work because of illness or medical circumstances.

Employees should inform their manager as soon as possible.

For extended absence, the company may request appropriate medical documentation where permitted.

------Match 2------
Employees should not work while seriously ill if doing so creates a health or workplace safety concern.

14. ANNUAL/EARNED LEAVE

Eligible employees receive annual paid leave according to company policy and applicable law.

Employees should submit planned leave requests in advance.



## TOOL


In [59]:
# create a retriever object so that it can invke when needed
retriever=vector_store.as_retriever(search_kwargs={"k":3})# return top 3

def search_In_VectorsDB(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chucks=retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chucks)


# PHASE-2: DATA RETRIEVAL PIPELINE

### LLM SETUP

In [60]:
from langchain_groq import ChatGroq
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0 #creativity
)
print(llm.model)

openai/gpt-oss-120b


In [61]:
response=llm.invoke("what is ai in 2 line")
print(response.content)

Artificial Intelligence (AI) is the field of computer science that creates systems capable of performing tasks that normally require human intelligence—such as learning, reasoning, perception, and decision‑making. In short, it enables machines to mimic and augment human cognitive abilities.


# AI AGENT
LLM=BRAIN

TOOL=RETRIEVER

MEMORY-NO MEMORY

In [62]:
from langchain.agents import create_agent

hr_assistant=create_agent(
    model=llm,
    tools=[search_In_VectorsDB],
    system_prompt=""" 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)
print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [63]:
def ask_assistant(question:str)->str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)
    response=hr_assistant.invoke({"messages":[{
        "role":"user","content":question
    }]})

    answer=response["messages"][-1].content
    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer


In [64]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)
print(response["messages"][-1].content)

I’m the friendly HR assistant here at **Acme Crop**. How can I help you today?


In [66]:
print(ask_assistant("how many hours have to work"))

QUESTION: how many hours have to work
------------------------------------------------------------
ANSWER: According to Acme Crop’s **Working Hours Policy**:

- **Full‑time employees** work **40 hours per week**, typically scheduled **Monday – Friday, 9:00 AM – 6:00 PM** (with a lunch break as arranged by the team).  
- **Part‑time employees** work **fewer than 40 hours per week**; the exact number depends on the individual’s employment agreement and team schedule.  

If you’re a full‑time employee, you’re expected to meet the 40‑hour weekly standard unless you have a different arrangement approved by your manager. If you’re part‑time, check your specific contract or speak with your manager for your exact weekly hours.

According to Acme Crop’s **Working Hours Policy**:

- **Full‑time employees** work **40 hours per week**, typically scheduled **Monday – Friday, 9:00 AM – 6:00 PM** (with a lunch break as arranged by the team).  
- **Part‑time employees** work **fewer than 40 hours per 